In [ ]:
# Upload this notebook into Colab directly. Keep REPO_DIR pointed at the folder in Google Drive.
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/colab_hc3_bundle"


In [ ]:
%cd "$REPO_DIR"
!python -m pip install -q --upgrade pip
!python -m pip install -q --upgrade -r requirements.txt


In [ ]:
import sys
from pathlib import Path

sys.path.append(REPO_DIR)

import torch
from colab_pipeline import (
    ColabExperimentConfig,
    run_baseline_reference,
    run_gptzero_experiment,
)

DATA_DIR = Path(REPO_DIR) / "artifacts" / "data" / "hc3"
SCORE_SPLITS = ("test",)
FIXED_FPR_TARGETS = (0.01, 0.0001)

MODEL_EVALUATION_DATA_FILES = {
    "svm_tfidf": {
        "train": DATA_DIR / "train.parquet",
        "val": DATA_DIR / "val.parquet",
        "test": DATA_DIR / "test.parquet",
    },
    "xgboost_tfidf": {
        "train": DATA_DIR / "train.parquet",
        "val": DATA_DIR / "val.parquet",
        "test": DATA_DIR / "test.parquet",
    },
    "gptzero_like": {
        "train": DATA_DIR / "train.parquet",
        "val": DATA_DIR / "val.parquet",
        "test": DATA_DIR / "test.parquet",
    },
}

for detector_name, split_paths in MODEL_EVALUATION_DATA_FILES.items():
    for split_name, split_path in split_paths.items():
        if not split_path.exists():
            raise FileNotFoundError(f"{detector_name} {split_name} data file not found: {split_path}")

print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("fixed FPR targets:", FIXED_FPR_TARGETS)
MODEL_EVALUATION_DATA_FILES


Baseline models. This section scores the existing SVM and XGBoost models on HC3. Use the next cell as-is to use the saved baseline models, or enable the optional retraining cell after it if you want to overwrite the saved baseline models and refresh the baseline run outputs.

In [ ]:
baseline_split_paths = MODEL_EVALUATION_DATA_FILES["svm_tfidf"]
if baseline_split_paths != MODEL_EVALUATION_DATA_FILES["xgboost_tfidf"]:
    raise ValueError(
        "SVM and XGBoost are scored together by run_baseline_reference, so their split files must match."
    )

baseline_config = ColabExperimentConfig(
    data_dir=baseline_split_paths["test"].parent,
    split_paths=baseline_split_paths,
    baseline_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "baselines",
    baseline_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_baselines_run",
    row_batch_size=1024,
    target_fpr=FIXED_FPR_TARGETS[0],
    target_fprs=FIXED_FPR_TARGETS,
)

baseline_result = run_baseline_reference(
    config=baseline_config,
    retrain=False,
    score_splits=SCORE_SPLITS,
)
baseline_result


In [ ]:
retrain_baselines = False

if retrain_baselines:
    baseline_retrain_result = run_baseline_reference(
        config=ColabExperimentConfig(
            data_dir=baseline_split_paths["test"].parent,
            split_paths=baseline_split_paths,
            baseline_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "baselines",
            baseline_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_baselines_run",
            row_batch_size=1024,
            xgb_batch_size=1024,
            xgb_device="cuda",
            target_fpr=FIXED_FPR_TARGETS[0],
            target_fprs=FIXED_FPR_TARGETS,
        ),
        retrain=True,
        score_splits=SCORE_SPLITS,
    )
    baseline_retrain_result
else:
    print("Skipping baseline retraining")


Optional: download a local GPT-2 scorer copy. Run this once with `REPO_DIR` pointing at this bundle if you want `hf_models/gpt2/` populated before uploading the folder to Google Drive.

In [ ]:
download_lm_dir = Path(REPO_DIR) / "hf_models" / "gpt2"
if download_lm_dir.exists():
    print(f"Local GPTZero scorer model already present at {download_lm_dir}")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    download_lm_dir.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.save_pretrained(download_lm_dir)
    scorer_model = AutoModelForCausalLM.from_pretrained("gpt2", use_safetensors=False)
    scorer_model.save_pretrained(download_lm_dir, safe_serialization=False)
    print(f"Saved local GPTZero scorer model to {download_lm_dir}")


GPTZero-like model. This section trains the GPTZero-like detector on HC3, then scores the test split. Rerunning it overwrites the saved detector under `artifacts/models/gptzero_like/` and refreshes `artifacts/runs/hc3_gptzero_run/`. It prefers the local `hf_models/gpt2/` copy when present, and falls back to downloading `gpt2` only if that folder is missing.

In [ ]:
local_lm_dir = Path(REPO_DIR) / "hf_models" / "gpt2"
if local_lm_dir.exists():
    lm_model = str(local_lm_dir)
    local_files_only = True
    print(f"Using local GPTZero scorer model: {lm_model}")
else:
    lm_model = "gpt2"
    local_files_only = False
    print("Local hf_models/gpt2 not found. Falling back to Hugging Face download for gpt2.")

gptzero_split_paths = MODEL_EVALUATION_DATA_FILES["gptzero_like"]
gptzero_config = ColabExperimentConfig(
    data_dir=gptzero_split_paths["test"].parent,
    split_paths=gptzero_split_paths,
    gptzero_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "gptzero_like",
    gptzero_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_gptzero_run",
    lm_model=lm_model,
    device="cuda",
    local_files_only=local_files_only,
    row_batch_size=64,
    perplexity_batch_size=16,
    max_sentences_per_text=None,
    target_fpr=FIXED_FPR_TARGETS[0],
    target_fprs=FIXED_FPR_TARGETS,
)

gptzero_result = run_gptzero_experiment(
    config=gptzero_config,
    train_model=True,
    score_splits=SCORE_SPLITS,
)
gptzero_result


Plotting. These cells load metrics from explicit per-detector CSV paths and generate comparison figures for the full ROC curve, low-FPR ROC behavior, summary metrics, and fixed-FPR TPR values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gpt_zero.metrics import fixed_fpr_metric_name, target_fpr_metric_name

baseline_metrics_dir = baseline_config.baseline_run_dir / "metrics"
gptzero_metrics_dir = gptzero_config.gptzero_run_dir / "metrics"

MODEL_METRIC_SOURCES = {
    "svm_tfidf": {
        "summary_path": baseline_metrics_dir / "metrics_summary.csv",
        "roc_path": baseline_metrics_dir / "roc_points.csv",
    },
    "xgboost_tfidf": {
        "summary_path": baseline_metrics_dir / "metrics_summary.csv",
        "roc_path": baseline_metrics_dir / "roc_points.csv",
    },
    "gptzero_like": {
        "summary_path": gptzero_metrics_dir / "metrics_summary.csv",
        "roc_path": gptzero_metrics_dir / "roc_points.csv",
    },
}

FPR_TARGET_LABELS = [
    (0.01, "1% FPR"),
    (0.0001, "0.01% FPR"),
]
FIXED_FPR_CLASSIFICATION_METRICS = ["accuracy", "precision", "recall", "f1"]


def _read_detector_metric_files(detector_name, sources):
    summary_path = Path(sources["summary_path"])
    roc_path = Path(sources["roc_path"])
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing summary file for {detector_name}: {summary_path}")
    if not roc_path.exists():
        raise FileNotFoundError(f"Missing ROC file for {detector_name}: {roc_path}")

    summary_frame = pd.read_csv(summary_path)
    roc_frame = pd.read_csv(roc_path)
    detector_summary = summary_frame[summary_frame["detector_name"] == detector_name].copy()
    detector_roc = roc_frame[roc_frame["detector_name"] == detector_name].copy()
    if detector_summary.empty:
        raise ValueError(f"No summary rows for {detector_name} in {summary_path}")
    if detector_roc.empty:
        raise ValueError(f"No ROC rows for {detector_name} in {roc_path}")

    detector_summary["metric_source"] = str(summary_path)
    detector_summary["roc_source"] = str(roc_path)
    detector_roc["roc_source"] = str(roc_path)
    return detector_summary, detector_roc


def _operating_point_from_roc_points(roc_group, target_fpr):
    valid = roc_group.loc[roc_group["fpr"] <= target_fpr].copy()
    if valid.empty:
        return None
    best_tpr = valid["tpr"].max()
    best = valid[np.isclose(valid["tpr"], best_tpr)].sort_values("fpr").iloc[-1]
    return best


def _fixed_fpr_values_from_roc(summary_row, roc_group, target_fpr):
    operating_point = _operating_point_from_roc_points(roc_group, target_fpr)
    if operating_point is None:
        return {}

    positives = float(summary_row["fn"] + summary_row["tp"])
    negatives = float(summary_row["tn"] + summary_row["fp"])
    tp = float(operating_point["tpr"]) * positives
    fp = float(operating_point["fpr"]) * negatives
    fn = positives - tp
    tn = negatives - fp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / positives if positives > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (positives + negatives) if (positives + negatives) > 0 else np.nan

    return {
        fixed_fpr_metric_name("threshold", target_fpr): float(operating_point["threshold"]),
        fixed_fpr_metric_name("actual_fpr", target_fpr): float(operating_point["fpr"]),
        target_fpr_metric_name(target_fpr): float(operating_point["tpr"]),
        fixed_fpr_metric_name("accuracy", target_fpr): accuracy,
        fixed_fpr_metric_name("precision", target_fpr): precision,
        fixed_fpr_metric_name("recall", target_fpr): recall,
        fixed_fpr_metric_name("f1", target_fpr): f1,
    }


def _add_fixed_fpr_columns(summary_frame, roc_frame):
    output = summary_frame.copy()
    for target_fpr, _ in FPR_TARGET_LABELS:
        metric_names = [target_fpr_metric_name(target_fpr)] + [
            fixed_fpr_metric_name(metric, target_fpr) for metric in FIXED_FPR_CLASSIFICATION_METRICS
        ]
        metric_names.extend(
            [fixed_fpr_metric_name("threshold", target_fpr), fixed_fpr_metric_name("actual_fpr", target_fpr)]
        )
        for metric_name in metric_names:
            if metric_name not in output.columns:
                output[metric_name] = np.nan
        for (detector_name, split_name), roc_group in roc_frame.groupby(["detector_name", "split"], dropna=False):
            mask = (output["detector_name"] == detector_name) & (output["split"] == split_name)
            for row_index, summary_row in output.loc[mask].iterrows():
                values = _fixed_fpr_values_from_roc(summary_row, roc_group, target_fpr)
                for metric_name, metric_value in values.items():
                    if pd.isna(output.at[row_index, metric_name]):
                        output.at[row_index, metric_name] = metric_value
    return output


summary_frames = []
roc_frames = []
for detector_name, sources in MODEL_METRIC_SOURCES.items():
    detector_summary, detector_roc = _read_detector_metric_files(detector_name, sources)
    summary_frames.append(detector_summary)
    roc_frames.append(detector_roc)

summary = pd.concat(summary_frames, ignore_index=True)
roc = pd.concat(roc_frames, ignore_index=True)
summary = _add_fixed_fpr_columns(summary, roc)
summary_test = summary[summary["split"] == "test"].copy()
roc_test = roc[roc["split"] == "test"].copy()

fixed_fpr_summary_columns = ["detector_name", "roc_auc"]
for target_fpr, _ in FPR_TARGET_LABELS:
    fixed_fpr_summary_columns.extend(
        [fixed_fpr_metric_name(metric, target_fpr) for metric in FIXED_FPR_CLASSIFICATION_METRICS]
    )
fixed_fpr_summary_columns.extend(["metric_source", "roc_source"])
summary_test[fixed_fpr_summary_columns]


In [ ]:
def _roc_xlim_for_target(target_fpr):
    lower = max(1e-6, target_fpr / 1000)
    upper = min(1.0, target_fpr * 1.2)
    return lower, upper


for target_fpr, label in FPR_TARGET_LABELS:
    plt.figure(figsize=(7, 6))

    for detector_name, group in roc_test.groupby("detector_name"):
        group = group.sort_values("fpr")
        auc_value = summary_test.loc[summary_test["detector_name"] == detector_name, "roc_auc"].iloc[0]
        plt.semilogx(
            group["fpr"].clip(lower=1e-6),
            group["tpr"],
            linewidth=2,
            label=f"{detector_name} (AUC={auc_value:.4f})",
        )

    plt.axvline(target_fpr, linestyle="--", linewidth=1.2, alpha=0.6, label=label)
    plt.xlim(*_roc_xlim_for_target(target_fpr))
    plt.ylim(0.0, 1.001)
    plt.xlabel("False Positive Rate (log scale)")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve on HC3 Test Split up to {label}")
    plt.legend()
    plt.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()


In [ ]:
for target_fpr, label in FPR_TARGET_LABELS:
    plt.figure(figsize=(7, 6))

    for detector_name, group in roc_test.groupby("detector_name"):
        group = group.sort_values("fpr")
        plt.semilogx(group["fpr"].clip(lower=1e-6), group["tpr"], linewidth=2, label=detector_name)

    plt.axvline(target_fpr, linestyle="--", linewidth=1.2, alpha=0.6, label=label)
    plt.xlim(max(1e-6, target_fpr / 100), min(1.0, target_fpr * 10))
    plt.ylim(0.0, 1.001)
    plt.xlabel("False Positive Rate (log scale)")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Detail Around {label} on HC3 Test Split")
    plt.legend()
    plt.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()


In [ ]:
detectors = summary_test["detector_name"].tolist()
x = np.arange(len(detectors))

for target_fpr, label in FPR_TARGET_LABELS:
    metrics_to_plot = ["accuracy", "f1", "roc_auc", target_fpr_metric_name(target_fpr)]
    labels = ["Accuracy@0.5", "F1@0.5", "ROC-AUC", f"TPR@{label}"]
    width = min(0.16, 0.8 / len(metrics_to_plot))

    fig, ax = plt.subplots(figsize=(11, 6))
    for i, (metric, metric_label) in enumerate(zip(metrics_to_plot, labels)):
        offset = (i - (len(metrics_to_plot) - 1) / 2) * width
        ax.bar(x + offset, summary_test[metric], width, label=metric_label)

    ax.set_xticks(x)
    ax.set_xticklabels(detectors, rotation=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"Test Metrics by Detector at {label}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
detectors = summary_test["detector_name"].tolist()
x = np.arange(len(detectors))
width = 0.8 / len(FIXED_FPR_CLASSIFICATION_METRICS)

for target_fpr, label in FPR_TARGET_LABELS:
    fig, ax = plt.subplots(figsize=(9, 6))
    for i, metric in enumerate(FIXED_FPR_CLASSIFICATION_METRICS):
        column = fixed_fpr_metric_name(metric, target_fpr)
        offset = (i - (len(FIXED_FPR_CLASSIFICATION_METRICS) - 1) / 2) * width
        ax.bar(x + offset, summary_test[column], width, label=metric.title())

    ax.set_xticks(x)
    ax.set_xticklabels(detectors, rotation=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"Classification Metrics at {label}")
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
